# MinecraftWorldModel 训练 Demo（Δz-JEPA · in-context 控制重映射）

从 OpenAI BASALT 离线演示数据（`.mp4` + `.jsonl`）训练 `MinecraftWorldModel`：
Δz-JEPA 前向预测 + 逆动力学 + SIGReg 防坍缩的自监督世界模型。本 demo 默认开启
**逐 episode 控制重映射**（`--control_remap`，档①「看视频掌握玩法」）。

**判断训练有效的指标**（不看 loss 绝对值）：
- `pred < 1.0`：世界模型预测误差低于 persistence（零运动）基线
- `kb_bal_acc > 0.6`：逆动力学能从 ΔZ 反推出键盘按键
- `mouse_move_acc > 0.2`：能区分鼠标是否在运动（恒中心 bin 基率 = 0）
- **`kb_onset_sub` ≫ chance 且随训练上升**：在**被重映射的 6 个键**上靠 context 学到了重绑定（in-context 核心信号）
- **`kb_sub_late > kb_sub_early`**：在用 context 适应（晚步上下文更多）

**安全提示**：密钥一律从 **Colab Secrets**（左侧 🔑）注入（见 §1），不再硬编码——本文件已脱敏、可入库。
clone 用的 PAT 若曾明文出现过，请去 github.com/settings/tokens 撤销/轮换；wandb key 同理（wandb.ai/authorize）。

In [ ]:
# ===== §1 配置 =====
import json, os, requests

BASE = "https://openaipublic.blob.core.windows.net/minecraft-rl"
INDEX = {
    "FindCave":      f"{BASE}/snapshots/find-cave-Jul-28.json",
    "MakeWaterfall": f"{BASE}/snapshots/waterfall-Jul-28.json",
    "AnimalPen":     f"{BASE}/snapshots/pen-animals-Jul-28.json",
    "BuildHouse":    f"{BASE}/snapshots/build-house-Jul-28.json",
}
TASKS = ["FindCave", "MakeWaterfall", "AnimalPen", "BuildHouse"]   # 四任务混采:
# 任务文本条件(text token)只有在任务间存在真实行为方差时才携带信息——单任务
# 数据下文本是常数(零互信息),模型会正确地无视它;混采同时给 plan 头坍缩意图多峰。

DISK_KEEP  = 64          # 训练池磁盘滑动窗口(段);64 段 ≈ 5GB。滚动下载器在四个
                         # 任务的**全量索引**上循环,旧的删掉防占满
DL_THREADS = 8           # 后台滚动下载线程数
SIZE       = 128         # 训练分辨率(VPTStreamDataset 解码时 on-the-fly resize)
CAM_SCALE  = 20.0        # 相机归一化尺度(**度**/帧;§2 已把原始像素×360/2400 转成度,
                         # 运动帧 |dx| 99% 分位 ≈ 18°)

# 密钥从 Colab Secrets(左侧 🔑 图标)注入,严禁硬编码入库:在 Secrets 建 GH_PAT / WANDB_API_KEY、
# 打开本 notebook 访问开关即可。曾明文出现过的 token 请去 settings/tokens 与 wandb.ai/authorize 撤销轮换。
try:
    from google.colab import userdata
    PAT       = userdata.get("GH_PAT")
    WANDB_KEY = userdata.get("WANDB_API_KEY")
except Exception:   
    from dotenv import load_dotenv                         # 非 Colab 环境回退到环境变量
    load_dotenv()
    PAT       = os.environ.get("GH_PAT", "")
    WANDB_KEY = os.environ.get("WANDB_API_KEY", "")
WANDB_PROJECT = "minecraft-world-model"     # wandb 项目名
WANDB_RUN    = None                         # run 名,None = 自动生成

REPO     = "OopsYouDiedE/tao-not-42-base-refactor-world-model-contract"
REPO_DIR = "/content/repo"
OUT      = "/content/runs/data"              # 数据根目录(train/ 滚动池 + holdout/ 固定)

ALL = {}                                    # task -> (basedir, relpaths)
for _t in TASKS:
    _idx = requests.get(INDEX[_t], timeout=60).json()
    ALL[_t] = (_idx["basedir"].rstrip("/"), _idx["relpaths"])
    print(f"  {_t}: {len(ALL[_t][1])} 段")
print(f"四任务混采  滑动窗口={DISK_KEEP} 段  holdout=每任务 1 段  CAM_SCALE={CAM_SCALE}")

In [ ]:
# ===== §2 数据准备:固定 holdout(每任务 1 段)+ 四任务全量索引滚动流式下载 =====
# 架构:后台 DL_THREADS 个线程在**四个任务的全量索引**上随机循环:下载 mp4 +
# 转换 jsonl(写入该任务的 task 文本,供 --text_encoder 条件化)→ OUT/train,
# 超过 DISK_KEEP 段按 mtime 删最老的(磁盘恒定 ~5GB)。训练 worker 每次换 clip
# 重扫目录、随机抽"当前已有"的段——下载快慢只影响轮换速度,**采样永不阻塞**。
# holdout = 每任务取索引末 1 段(共 4 段)下载到 OUT/holdout,固定不动。
#
# 并发竞态防护(多线程下载 × 逐出 × 训练 worker 读取同时进行):
#   - 文件大小/mtime 查询全部容错(查询瞬间文件可能被兄弟线程逐出);
#   - 临时文件名带线程 id:两线程撞中同一段时各写各的,os.replace 原子,
#     后到者覆盖,内容不互相腐蚀;
#   - 训练侧 VPTStreamDataset 对缺文件/半写文件静默换段重试(仓库端已实现)。
#
# jsonl 转换契约(与 utils/vpt_dataset._action_vec 严格一致):
#   keyboard.keys 列表 → {key_*: 1};鼠标按钮 0/1 → key_attack/key_use;
#   dx/dy 原始是**像素**,×360/2400 转成度(CAM_SCALE=20 按度标定);
#   isGuiOpen 帧 dx/dy 置 0、按钮不映射,并写 "gui":1 标记——GUI 内画面变化
#   (光标/物品)与记录动作零相关,训练侧据此**拒采**这类窗口(标签噪声)。
import threading, random
from concurrent.futures import ThreadPoolExecutor

OUT_TRAIN = f"{OUT}/train"
OUT_HOLD  = f"{OUT}/holdout"
os.makedirs(OUT_TRAIN, exist_ok=True); os.makedirs(OUT_HOLD, exist_ok=True)

# 旧布局清理:根目录散落的 mp4/jsonl/临时文件是旧版的,新布局读不到,删掉防占盘
for f in os.listdir(OUT):
    p = os.path.join(OUT, f)
    if os.path.isfile(p) and (f.endswith((".mp4", ".jsonl")) or ".part" in f):
        os.remove(p)

PX2DEG = 360.0 / 2400.0          # VPT CAMERA_SCALER:鼠标像素 → 视角度数

_KEYMAP = {
    "key.keyboard.w": "key_w", "key.keyboard.s": "key_s",
    "key.keyboard.a": "key_a", "key.keyboard.d": "key_d",
    "key.keyboard.space": "key_space",
    "key.keyboard.left.shift": "key_sneak",
    "key.keyboard.left.control": "key_sprint",
    "key.keyboard.q": "key_drop", "key.keyboard.e": "key_inventory",
    **{f"key.keyboard.{i}": f"key_hotbar.{i}" for i in range(1, 10)},
}
_BTN = {0: "key_attack", 1: "key_use"}
TASK_TEXT = {"FindCave": "find a cave", "MakeWaterfall": "make a waterfall",
             "AnimalPen": "build an animal pen", "BuildHouse": "build a village house"}

def _size(p):
    """容错 getsize:查询瞬间文件可能被兄弟线程逐出 → 视作不存在。"""
    try:
        return os.path.getsize(p)
    except OSError:
        return -1

def _mtime(p):
    try:
        return os.path.getmtime(p)
    except OSError:
        return 0.0            # 已被删:排到最老,_evict 的 remove 本就容错

def prepare(task, rel, out_dir):
    """下载一段 mp4 + jsonl 到 out_dir;jsonl 同步转键名/单位并写入该任务文本。
    临时名带线程 id + os.replace 原子落盘:中断/竞态/同段双下都安全。"""
    basedir = ALL[task][0]
    name = os.path.basename(rel)[:-4]
    mp4, jsonl = f"{out_dir}/{name}.mp4", f"{out_dir}/{name}.jsonl"
    tmp = f".part{threading.get_ident()}"

    if _size(mp4) <= 0:                                             # mp4:流式直存
        r = requests.get(f"{basedir}/{rel}", stream=True, timeout=300); r.raise_for_status()
        with open(mp4 + tmp, "wb") as f:
            for chunk in r.iter_content(1 << 22):
                f.write(chunk)
        os.replace(mp4 + tmp, mp4)

    if not os.path.exists(jsonl):                                   # jsonl:整取 + 转键名
        r = requests.get(f"{basedir}/{rel[:-4]}.jsonl", timeout=300); r.raise_for_status()
        with open(jsonl + tmp, "w", encoding="utf-8") as fw:
            for i, line in enumerate(r.text.splitlines()):
                d = json.loads(line)
                gui = bool(d.get("isGuiOpen", False))
                kb = {_KEYMAP[k]: 1 for k in d.get("keyboard", {}).get("keys", []) if k in _KEYMAP}
                m = d.get("mouse", {})
                if not gui:
                    kb.update({_BTN[b]: 1 for b in m.get("buttons", []) if b in _BTN})
                dx = 0.0 if gui else m.get("dx", 0.0) * PX2DEG
                dy = 0.0 if gui else m.get("dy", 0.0) * PX2DEG
                frame = {"keyboard": kb, "mouse": {"dx": dx, "dy": dy}}
                if gui:
                    frame["gui"] = 1          # GUI 段标记:训练侧据此拒采(标签噪声)
                if i == 0:
                    frame["task"] = TASK_TEXT[task]
                fw.write(json.dumps(frame, ensure_ascii=False) + "\n")
        os.replace(jsonl + tmp, jsonl)
    return name

# --- holdout:每任务取索引末 1 段,固定不动(同步下载,~320MB) ---
HOLDOUT = [(t, ALL[t][1][-1]) for t in TASKS]
with ThreadPoolExecutor(max_workers=4) as ex:
    list(ex.map(lambda tr: prepare(tr[0], tr[1], OUT_HOLD), HOLDOUT))
print(f"holdout ready: {len(HOLDOUT)} clips(每任务 1 段)in {OUT_HOLD}")

# --- train:后台滚动下载(四任务全量索引循环,daemon 线程,§3 训练期间持续运行) ---
_HOLD_SET = {rel for _, rel in HOLDOUT}
POOL = [(t, rel) for t in TASKS for rel in ALL[t][1] if rel not in _HOLD_SET]

def _evict():
    """训练池超过 DISK_KEEP 段 → 按 mtime 删最老的 mp4+jsonl(滑动窗口)。"""
    mp4s = sorted((os.path.join(OUT_TRAIN, f) for f in os.listdir(OUT_TRAIN)
                   if f.endswith(".mp4")), key=_mtime)
    for old in mp4s[:max(0, len(mp4s) - DISK_KEEP)]:
        for p in (old, old[:-4] + ".jsonl"):
            try:
                os.remove(p)
            except OSError:
                pass                         # 兄弟线程已删/训练 worker 占用:下轮再删

def _roller(tid):
    rng = random.Random(1234 + tid)
    while True:
        try:
            t, rel = POOL[rng.randrange(len(POOL))]
            prepare(t, rel, OUT_TRAIN)
            _evict()
        except Exception as e:
            print(f"[dl-{tid}] {type(e).__name__}: {e}")   # 单段失败不终止滚动

if not globals().get("_DL_STARTED"):       # 幂等:重跑本 cell 不重复起线程
    _DL_STARTED = True
    for i in range(DL_THREADS):
        threading.Thread(target=_roller, args=(i,), daemon=True).start()
n_now = sum(f.endswith('.mp4') for f in os.listdir(OUT_TRAIN))
print(f"rolling downloader: {DL_THREADS} 线程在 {len(POOL)} 段(四任务)索引上循环 → "
      f"{OUT_TRAIN}(磁盘保留 {DISK_KEEP} 段,当前 {n_now} 段)。可直接运行 §3。")
print("⚠ 旧数据的 jsonl 没有 gui 标记:GUI 过滤只对本次之后下载的段生效;"
      "要全量生效,删 train/ 下旧 jsonl 让滚动器重转(mp4 复用)。")

In [ ]:
# ===== §3 训练 · in-context 控制重映射(看视频掌握玩法,档①)=====
# 逐 episode 把 6 个高信号键(w/a/s/d/space/attack)随机置换、video(画面)不动:模型只能从
# window 早期步 in-context 辨识出本局控制映射 T 才能预测/反推动作。train/eval 用 disjoint 置换集
# ⇒ holdout 表现量的是「靠观察掌握没见过的控制方案」,不是记忆。inv_dyn_ctx(FiLM-on-h)是重绑定
# 硬前提,已是 net.config 默认。骨干用开放权重 dinov2(--config dinov2.yaml,无需 HF token)。
#
# 读法(eval 面板 [eval] 行):kb_onset_sub ≫ chance 且随训练上升 = 学到重绑定(6 个被置换键靠
# context 推出语义);kb_sub_late > kb_sub_early = 在用 context 适应;kb_*_sub vs kb_*_rest 差距
# = remap 代价/推断是否补齐恒等键水平。best 用 pred_move(世界模型整体质量,稳);想让 best 直接
# 跟重绑定走,改 --best_metric kb_onset_sub。
import os,wandb,glob

if not os.path.exists(os.path.join(REPO_DIR, '.git')):
    !rm -rf {REPO_DIR}
    !git clone --depth=1 https://{PAT}@github.com/{REPO}.git {REPO_DIR}
else:
    !cd {REPO_DIR} && git fetch --depth=1 origin main && git reset --hard origin/main
!cd {REPO_DIR} && git log --oneline -1
# 拉代码后按 requirements 同步依赖。剔除 torch/torchvision/xformers:与 Colab 预装 CUDA torch
# 强耦合(xformers 会按 torch 版本拉特定 build、可能反向降级 GPU torch)。无 --upgrade ⇒ 已装的
# tensorflow/scipy/opencv 跳过,只补 wandb/transformers 等。
!cd {REPO_DIR} && grep -ivE '^(torch|torchvision|xformers)' requirements.txt > /tmp/req_colab.txt && pip install -q -r /tmp/req_colab.txt
os.environ["WANDB_API_KEY"] = WANDB_KEY

art = wandb.Api().artifact(f"{WANDB_PROJECT}/best-incontext-remap6-128px:latest", type="model")
ckpt = glob.glob(f"{art.download(root=f'{REPO_DIR}/runs/mc_ckpt')}/**/*.pt", recursive=True)[0]
print("warm-start from:", ckpt)
!cd {REPO_DIR} && python train/minecraft/train_minecraft.py \
  --init_from "{ckpt}" \
  --config configs/minecraft/dinov2.yaml \
  --data_dir {OUT}/train \
  --holdout_dir {OUT}/holdout \
  --control_remap \
  --text_encoder minilm \
  --camera_scale {CAM_SCALE} \
  --img_size 128 \
  --batch 128 \
  --seq_len 60 \
  --frame_skip 8 \
  --steps_per_epoch 5 \
  --epochs 300 \
  --ema_decay 0.97 \
  --rho_open 0 \
  --beta_kl 0.1 \
  --kl_free 1.0 \
  --open_k 4 \
  --mouse_move_w 4 \
  --gamma_plan 0.25 \
  --beta_sigreg 0.03 \
  --beta_div 0.05 \
  --log_every 1 \
  --viz_every 40 \
  --eval_every 20 \
  --clip_cache 4 \
  --clip_refresh 256 \
  --save_best_every 50 \
  --best_metric pred_move \
  --ckpt_dir runs/mc_ckpt \
  --viz_dir runs/mc_viz_128 \
  --wandb \
  --wandb_project {WANDB_PROJECT} \
  --wandb_run incontext-remap6-128px \
  --device cuda
